In [2]:
import duckdb
import os

con = duckdb.connect("../data/mot.duckdb")

In [3]:
con.sql("SHOW TABLES").show()

┌───────────────────┐
│       name        │
│      varchar      │
├───────────────────┤
│ items_dedup       │
│ results_dedup     │
│ test_conflicts    │
│ test_item_clean   │
│ test_result_clean │
└───────────────────┘



In [4]:
con.sql("SELECT COUNT(*) FROM test_result_clean").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     42430899 │
└──────────────┘



Now lets look for the lookup tables in the data:

In [5]:
lookup_folder = "../data/raw/lookup"
lookup_files = sorted(f for f in os.listdir(lookup_folder) if f.endswith((".csv", ".txt")))
lookup_files

['dft_item_detail_extract.csv',
 'dft_item_group_extract.csv',
 'dft_mdr_fuel_type_extract.csv',
 'dft_mdr_rfr_location_extract.csv',
 'dft_mdr_test_outcome_extract.csv',
 'dft_mdr_test_type_extract.csv']

Peek into every file:

In [6]:
for f in lookup_files:
    path = f"{lookup_folder}/{f}"
    print("=====", f)
    con.sql(f"SELECT * FROM read_csv('{path}', all_varchar=true) LIMIT 3").show(max_width=250)
    n = con.sql(f"SELECT COUNT(*) FROM read_csv('{path}', all_varchar=true)").fetchall()[0][0]
    print(n, "rows")

===== dft_item_detail_extract.csv
┌─────────┬───────────────┬──────────────┬────────────┬─────────────────────────┬───────────────────────────────┬────────────────┬────────────────────────────────────────────────────────┬───────────────────────────────────┬──────────────────────────┐
│ rfr_id  │ test_class_id │ test_item_id │ minor_item │ rfr_deficiency_category │           rfr_desc            │ rfr_loc_marker │                  rfr_insp_manual_desc                  │         rfr_advisory_text         │ test_item_set_section_id │
│ varchar │    varchar    │   varchar    │  varchar   │         varchar         │            varchar            │    varchar     │                        varchar                         │              varchar              │         varchar          │
├─────────┼───────────────┼──────────────┼────────────┼─────────────────────────┼───────────────────────────────┼────────────────┼────────────────────────────────────────────────────────┼──────────────────────────

Load all six files into a table:

In [8]:
lookup_tables = {
    "dft_item_detail_extract.csv":       "lookup_item_detail",
    "dft_item_group_extract.csv":        "lookup_item_group",
    "dft_mdr_fuel_type_extract.csv":     "lookup_fuel_type",
    "dft_mdr_rfr_location_extract.csv":  "lookup_location",
    "dft_mdr_test_outcome_extract.csv":  "lookup_test_outcome",
    "dft_mdr_test_type_extract.csv":     "lookup_test_type",
}

for file_name, table_name in lookup_tables.items():
    con.execute(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT * FROM read_csv('{lookup_folder}/{file_name}', all_varchar=true)
    """)
    n = con.sql(f"SELECT COUNT(*) FROM {table_name}").fetchall()[0][0]
    print(f"{table_name}: {n:,} rows")

lookup_item_detail: 21,105 rows
lookup_item_group: 4,144 rows
lookup_fuel_type: 14 rows
lookup_location: 129 rows
lookup_test_outcome: 7 rows
lookup_test_type: 6 rows


Check each lookup key is unique:

In [9]:
con.sql("""
    SELECT 'item_detail' AS tbl,
           COUNT(*) AS rows_total,
           COUNT(DISTINCT (rfr_id, test_class_id)) AS distinct_keys
    FROM lookup_item_detail
    UNION ALL
    SELECT 'item_group', COUNT(*), COUNT(DISTINCT (test_item_id, test_class_id))
    FROM lookup_item_group
""").show()

┌─────────────┬────────────┬───────────────┐
│     tbl     │ rows_total │ distinct_keys │
│   varchar   │   int64    │     int64     │
├─────────────┼────────────┼───────────────┤
│ item_detail │      21105 │         21105 │
│ item_group  │       4144 │          4144 │
└─────────────┴────────────┴───────────────┘



What are the defect categories?

In [10]:
con.sql("""
    SELECT rfr_deficiency_category, COUNT(*) AS n
    FROM lookup_item_detail
    GROUP BY rfr_deficiency_category
    ORDER BY n DESC
""").show()

┌─────────────────────────┬───────┐
│ rfr_deficiency_category │   n   │
│         varchar         │ int64 │
├─────────────────────────┼───────┤
│ Pre-EU Directive        │ 12595 │
│ Major                   │  5266 │
│ Dangerous               │  2370 │
│ Minor                   │   874 │
└─────────────────────────┴───────┘



What are the top-level groups for cars (class 4)?

In [11]:
con.sql("""
    SELECT test_item_id, item_name
    FROM lookup_item_group
    WHERE test_class_id = '4' AND parent_id = '0' AND test_item_id <> '0'
    ORDER BY TRY_CAST(test_item_id AS INTEGER)
""").show(max_rows=50)

┌──────────────┬────────────────────────────────────────────────┐
│ test_item_id │                   item_name                    │
│   varchar    │                    varchar                     │
├──────────────┼────────────────────────────────────────────────┤
│ 501          │ Towbars                                        │
│ 5000         │ Lamps, Reflectors and Electrical Equipment     │
│ 5100         │ Steering                                       │
│ 5190         │ Suspension                                     │
│ 5430         │ Brakes                                         │
│ 5650         │ Tyres                                          │
│ 5670         │ Road Wheels                                    │
│ 5680         │ Seat Belts and Supplementary Restraint Systems │
│ 5690         │ Body, Structure and General Items              │
│ 5730         │ Exhaust, Fuel and Emissions                    │
│ 5750         │ Driver's View of the Road                      │
│ 5780    

Does each defect’s section ID land on a group?

In [12]:
con.sql("""
    SELECT COUNT(*)              AS detail_rows,
           COUNT(g.test_item_id) AS matched_to_group
    FROM lookup_item_detail d
    LEFT JOIN lookup_item_group g
           ON d.test_item_set_section_id = g.test_item_id
          AND d.test_class_id = g.test_class_id
""").show()

┌─────────────┬──────────────────┐
│ detail_rows │ matched_to_group │
│    int64    │      int64       │
├─────────────┼──────────────────┤
│       21105 │            21105 │
└─────────────┴──────────────────┘



Label every item with its category:

In [14]:
con.execute("""
    CREATE OR REPLACE TABLE test_item_labelled AS
    SELECT i.test_id,
           i.rfr_id,
           i.rfr_type_code,
           i.location_id,
           i.dangerous_mark,
           r.test_class_id,
           d.rfr_deficiency_category,
           TRY_CAST(d.test_item_set_section_id AS INTEGER) AS section_id,
           g.item_name                                     AS section_name
    FROM test_item_clean i
    JOIN test_result_clean r
      ON i.test_id = r.test_id
    LEFT JOIN lookup_item_detail d
      ON i.rfr_id = TRY_CAST(d.rfr_id AS INTEGER)
     AND r.test_class_id = d.test_class_id
    LEFT JOIN lookup_item_group g
      ON d.test_item_set_section_id = g.test_item_id
     AND d.test_class_id = g.test_class_id
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Was every item kept?

In [15]:
con.sql("""
    SELECT COUNT(*)                        AS item_rows,
           COUNT(rfr_deficiency_category)  AS matched_detail,
           COUNT(section_name)             AS matched_group
    FROM test_item_labelled
""").show()

┌───────────┬────────────────┬───────────────┐
│ item_rows │ matched_detail │ matched_group │
│   int64   │     int64      │     int64     │
├───────────┼────────────────┼───────────────┤
│  91487420 │       91484721 │      91484721 │
└───────────┴────────────────┴───────────────┘



Which category does each item type carry, and what does D mean?

In [16]:
con.sql("""
    SELECT rfr_type_code, rfr_deficiency_category, dangerous_mark, COUNT(*) AS n
    FROM test_item_labelled
    GROUP BY rfr_type_code, rfr_deficiency_category, dangerous_mark
    ORDER BY rfr_type_code, n DESC
""").show(max_rows=60)

┌───────────────┬─────────────────────────┬────────────────┬──────────┐
│ rfr_type_code │ rfr_deficiency_category │ dangerous_mark │    n     │
│    varchar    │         varchar         │    varchar     │  int64   │
├───────────────┼─────────────────────────┼────────────────┼──────────┤
│ A             │ Major                   │ NULL           │ 30810968 │
│ A             │ Dangerous               │ NULL           │ 26182206 │
│ A             │ Pre-EU Directive        │ NULL           │  3151768 │
│ A             │ Minor                   │ NULL           │    63556 │
│ A             │ Pre-EU Directive        │ D              │    10387 │
│ A             │ NULL                    │ NULL           │        6 │
│ F             │ Major                   │ NULL           │ 17643656 │
│ F             │ Dangerous               │ D              │  3714394 │
│ F             │ NULL                    │ NULL           │     2388 │
│ F             │ NULL                    │ D              │    

Which top-level groups do real failing items fall in?

In [17]:
con.sql("""
    SELECT section_id, section_name, COUNT(*) AS items
    FROM test_item_labelled
    WHERE test_class_id = '4' AND rfr_type_code IN ('F', 'P')
    GROUP BY section_id, section_name
    ORDER BY items DESC
""").show(max_rows=40)

┌────────────┬────────────────────────────────────────────────┬─────────┐
│ section_id │                  section_name                  │  items  │
│   int32    │                    varchar                     │  int64  │
├────────────┼────────────────────────────────────────────────┼─────────┤
│      20158 │ Lamps, reflectors and electrical equipment     │ 5708514 │
│      20218 │ Suspension                                     │ 4386900 │
│      20003 │ Brakes                                         │ 3730469 │
│      20449 │ Tyres                                          │ 2926764 │
│      20147 │ Visibility                                     │ 1833506 │
│      20310 │ Body, chassis, structure                       │ 1436248 │
│      20385 │ Noise, emissions and leaks                     │ 1319548 │
│      20106 │ Steering                                       │  789472 │
│      20366 │ Seat belts and supplementary restraint systems │  366878 │
│      20448 │ Road Wheels            

In [13]:
#con.close()